In [31]:
# import libraries for sentiment analysis
import pandas as pd # read dataset 
import numpy as np # numeric operations
from textblob import TextBlob # get subjectivity for each text
import re # text cleaning
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # get VADER scores for each text
from sklearn.model_selection import train_test_split # split data into train and test sets
from sklearn.metrics import accuracy_score, classification_report # evaluate model performance
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # classifier to predict sentiment labels

from newsapi import NewsApiClient
import pandas as pd
import yfinance as yf


In [32]:
import nltk 
nltk.download('vader_lexicon')  # to make the library work well for english 

import pandas as pd # read dataset 
import numpy as np # numeric operations
import matplotlib.pyplot as plt # plotting


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\35387\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [33]:
from newsapi import NewsApiClient
from datetime import date, timedelta, datetime
from nltk.sentiment.vader import SentimentIntensityAnalyzer 
sia = SentimentIntensityAnalyzer()

In [34]:
pd.set_option('display.max_colwidth', 1000)  # display all columns when printing a dataframe

In [35]:
from dotenv import load_dotenv
import os

# load .env in the current working directory (or give full path: load_dotenv('/path/to/.env'))
load_dotenv('.env')

# read the key from the .env file 
NEWS_API_KEY = os.environ['NEWS_API_KEY']

# print(NEWS_API_KEY)

In [36]:
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

keywords = "dow jones"
my_date = datetime.strptime('18-10-2025', '%d-%m-%Y').date()

from_date_str = my_date.isoformat()
to_date_str = (my_date + timedelta(days=30)).isoformat()

articles = newsapi.get_everything(
    q=keywords,
    from_param=from_date_str,
    to=to_date_str,
    language='en',
    sort_by='relevancy',
    page_size=100
)

articles_list = articles['articles']

df_articles = pd.DataFrame(articles_list)
df_articles['Date'] = pd.to_datetime(df_articles['publishedAt']).dt.date

grouped = df_articles.groupby('Date')['title'].apply(list).reset_index()

def top5(lst):
    lst = lst or []
    return (lst + [""] * 5)[:5]

tops = grouped['title'].apply(top5).tolist()

df1 = pd.DataFrame({
    "Date": grouped['Date'],
    "Label": 0,
    "Top1": [t[0] for t in tops],
    "Top2": [t[1] for t in tops],
    "Top3": [t[2] for t in tops],
    "Top4": [t[3] for t in tops],
    "Top5": [t[4] for t in tops],
})

df1.head()
# df = pd.DataFrame(articles)
# df.head()

,Date,Label,Top1,Top2,Top3,Top4,Top5
0,2025-10-19,0,"The Donald Trump Administration Purchased Stakes in Intel, MP Materials, Lithium Americas, and Trilogy Metals -- and It Sets a Dangerous Precedent",,,,
1,2025-10-20,0,Stocks Poised For More Gains As Shutdown Continues,"Stock market today: Dow, S&P 500, Nasdaq futures stall as investors eye earnings ahead",Gold is smashing records as the dollar wavers — prompting precious-metal miners to come off the sidelines,,
2,2025-10-21,0,Trump seeks to proceed with $10B lawsuit over WSJ story on Epstein's birthday book,Stock Futures Fall as Markets Brace for Key Earnings,"Stock market today: Dow, S&P 500, Nasdaq futures trade flat as investors look to Tesla earnings after Netflix disappoints","Stock market today: Dow, S&P 500, Nasdaq futures hit pause as investors brace for Tesla earnings after Netflix disappoints","Stock market today: Dow, S&P 500, Nasdaq futures wobble as next rush of earnings kicks off"
3,2025-10-22,0,Stocks Retreat on Chipmaker Weakness and Renewed China Tensions,Stock Market Today: Dow Falls As Netflix Dives On Earnings Miss; Tesla Results Next (Live Coverage),Investors Pause To Wait For Magnificent 7 Earnings,,
4,2025-10-23,0,Stock Futures Down After Tesla Earnings Miss,,,,


In [37]:
df = yf.download("^DJI", period="1mo", progress=False)
df.index.name = "Date"


C:\Users\35387\AppData\Local\Temp\ipykernel_39748\1968648038.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("^DJI", period="1mo", progress=False)


In [40]:
if isinstance(df.columns, pd.MultiIndex):
    # if '^DJI' is present select that tickers columns
    if '^DJI' in df.columns.get_level_values(1):
        df = df.xs('^DJI', axis=1, level=1)
    else:
        # otherwise drop the top level field and keep the first level names
        df.columns = df.columns.get_level_values(0)

df2 = df.reset_index()
cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']
df2 = df2[[c for c in cols if c in df.columns]]
df2

Price,Date,Open,High,Low,Close,Volume
0,2025-10-20,46312.878906,46759.269531,46312.878906,46706.578125,444010000
1,2025-10-21,46707.078125,47125.660156,46688.250000,46924.738281,430490000
2,2025-10-22,46941.558594,46941.558594,46461.519531,46590.410156,458450000
3,2025-10-23,46519.128906,46802.148438,46490.058594,46734.609375,407570000
4,2025-10-24,46811.511719,47326.730469,46811.511719,47207.121094,403500000
5,2025-10-27,47412.800781,47564.519531,47375.960938,47544.589844,426790000
6,2025-10-28,47752.351562,47943.160156,47675.699219,47706.371094,603670000
7,2025-10-29,47746.789062,48040.640625,47448.589844,47632.000000,683470000
8,2025-10-30,47446.878906,48014.921875,47381.910156,47522.121094,593890000
9,2025-10-31,47659.960938,47718.378906,47347.281250,47562.871094,704620000
